# VehiAlpes — Capa Silver

MINE-4214 · Taller 1 · Punto 2 (ELT con arquitectura Medallón)

## Objetivo de la capa

Entregar datos tipados, limpios y conciliados, todavía cercanos a la estructura de
la fuente y sin llaves surrogadas. Aquí vive **todo** el trabajo de calidad.

## Orden de ejecución

El notebook corre de arriba abajo sin intervención. El orden no es arbitrario:

| Paso | Qué produce | Por qué va en esa posición |
|---|---|---|
| 1 | Funciones de parseo | Las usan tanto transacciones como carros |
| 2 | `stg_transacciones` | Tipado, sin limpieza todavía |
| 3 | `dim_vehiculo_vigencias` | **Debe ir antes que la reparación**: para reconstruir una fecha se necesita la tarifa vigente |
| 4 | Deduplicación | Sobre datos ya tipados |
| 5 | Reparación y banderas | Consume las vigencias del paso 3 |
| 6 | `clientes` | Los miembros inferidos se derivan de las transacciones ya limpias |
| 7 | Conciliación | Verifica que la capa no perdió ni inventó filas |

In [0]:
from pyspark.sql import functions as F, Window as W

CATALOGO = "vehialpes"
FECHA_ALTA = "9999-12-31"

## Paso 1 — Estandarización de fechas

El problema más severo de la fuente. `fecha_inicio` y `fecha_fin` traen cuatro
formatos:

| Formato | Patrón | Ejemplo | Proporción |
|---|---|---|---|
| ISO | `AAAA-MM-DD` | `2024-08-26` | 55,5% |
| Día primero, barra | `DD/MM/AAAA` | `18/05/2024` | 19,2% |
| Mes primero, guion | `MM-DD-AAAA` | `05-18-2024` | 14,8% |
| Mes en español | `DD-mmm-AAAA` | `07-ago-2024` | 10,5% |

### Decisión: cada formato se identifica por expresión regular antes de parsear

Un `coalesce` de intentos sucesivos sin guardas **produce errores silenciosos**.
El caso concreto: si la rama del mes en español traduce el texto y luego aplica
`dd-MM-yyyy` a cualquier cadena con guiones, entonces `08-07-2024` —que en esta
fuente es 7 de agosto en formato mes primero— se leería como 8 de julio. El error
es indetectable porque ambas fechas son válidas.

Con guardas por patrón cada cadena entra a exactamente una rama, y cualquier valor
que no encaje en ninguna queda nulo y lo captura la validación.

### Decisión: `try_to_date` en lugar de `to_date`

Con ANSI activo, `to_date` lanza excepción ante un valor no parseable y aborta el
job. `try_to_date` devuelve nulo, lo que permite que la validación posterior reporte
**cuántas y cuáles** filas fallaron en lugar de morir en la primera.

### Cómo se resolvió la ambigüedad entre `DD/MM` y `MM-DD`

Los dos formatos intermedios son indistinguibles fila por fila. Se resolvió
analizando el rango de valores de cada posición sobre el conjunto completo: en el
formato con barra el primer componente llega hasta 31 y el segundo hasta 12, luego
es día primero; en el formato con guion ocurre lo contrario. Es una inferencia
estadística sobre la muestra, no un dato de la fuente, y por eso se documenta como
supuesto.

In [0]:
MESES_ES = {
    "ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06",
    "jul": "07", "ago": "08", "sep": "09", "oct": "10", "nov": "11", "dic": "12",
}

RX_ISO = r"^\d{4}-\d{2}-\d{2}$"
RX_BARRA = r"^\d{1,2}/\d{1,2}/\d{4}$"
RX_ESPANOL = r"(?i)^\d{1,2}-(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)-\d{4}$"
RX_GUION_NUM = r"^\d{1,2}-\d{1,2}-\d{4}$"


def vacio_a_nulo(col):
    """Convierte cadena vacía en nulo. Se evita `nullif` por compatibilidad de versión."""
    limpio = F.trim(col)
    return F.when(limpio == "", None).otherwise(limpio)


def _traducir_mes(col):
    resultado = F.lower(col)
    for abrev, numero in MESES_ES.items():
        resultado = F.regexp_replace(resultado, f"-{abrev}-", f"-{numero}-")
    return resultado


def _intentar_fecha(expresion_sql, patron):
    """`try_to_timestamp` devuelve nulo ante un valor no parseable en lugar de lanzar
    excepción. Se prefiere sobre `try_to_date`, que solo existe en Databricks SQL y no
    en Spark OSS, para que el notebook sea portable y probable fuera del workspace."""
    return F.expr(f"try_to_timestamp({expresion_sql}, '{patron}')").cast("date")


def parsear_fecha(nombre_columna):
    """Resuelve los cuatro formatos. Cada rama está protegida por su patrón, de modo
    que una cadena entra a una sola y nunca se interpreta con el formato equivocado."""
    col = vacio_a_nulo(F.col(nombre_columna))
    crudo = f"trim(`{nombre_columna}`)"
    espanol = _traducir_mes(col)
    return (F.when(col.rlike(RX_ISO), _intentar_fecha(crudo, "yyyy-MM-dd"))
             .when(col.rlike(RX_BARRA), _intentar_fecha(crudo, "dd/MM/yyyy"))
             .when(col.rlike(RX_ESPANOL),
                   F.try_to_timestamp(espanol, F.lit("dd-MM-yyyy")).cast("date"))
             .when(col.rlike(RX_GUION_NUM), _intentar_fecha(crudo, "MM-dd-yyyy"))
             .otherwise(None))


def a_numero(nombre_columna, tipo="int"):
    """Casteo seguro. Con ANSI activo —el default en DBR 14+ y en serverless—
    `cast('' as int)` **lanza excepción y aborta el job**. Las columnas numéricas de
    esta fuente traen cadenas vacías en la cola defectuosa del archivo, así que el
    casteo directo no es viable. `try_cast` devuelve nulo y deja que las reglas de
    calidad decidan qué hacer."""
    return F.expr(f"try_cast(trim(`{nombre_columna}`) as {tipo})")

## Paso 2 — `stg_transacciones`: tipado

Se castea a tipos definitivos y se conserva la columna original de cada fecha en
`*_original`. Esa retención permite auditar el parseo sin volver a bronze y es lo
que hace verificable la inferencia del formato ambiguo.

In [0]:
stg_transacciones = (spark.table(f"{CATALOGO}.bronze.transacciones")
    .withColumn("id_transaccion", a_numero("id_transaccion", "int"))
    .withColumn("tipo_transaccion", F.upper(F.trim(F.col("tipo_transaccion"))))
    .withColumn("fecha_inicio_original", F.col("fecha_inicio"))
    .withColumn("fecha_fin_original", F.col("fecha_fin"))
    .withColumn("fecha_inicio", parsear_fecha("fecha_inicio"))
    .withColumn("fecha_fin", parsear_fecha("fecha_fin"))
    .withColumn("placa", F.upper(F.trim(F.col("placa"))))
    .withColumn("id_cliente", a_numero("id_cliente", "int"))
    .withColumn("nombre_proveedor", vacio_a_nulo(F.col("nombre_proveedor")))
    .withColumn("nombre_sucursal", F.trim(F.col("nombre_sucursal")))
    .withColumn("ciudad", F.trim(F.col("ciudad")))
    .withColumn("metodo_pago", F.trim(F.col("metodo_pago")))
    .withColumn("km_transaccion", a_numero("km_transaccion", "int"))
    .withColumn("kms_recorridos", a_numero("kms_recorridos", "int"))
    .withColumn("valor_total", a_numero("valor_total", "bigint"))
    .withColumn("valor_seguro", a_numero("valor_seguro", "int"))
    .drop("_rescued_data"))

stg_transacciones.write.mode("overwrite").saveAsTable(f"{CATALOGO}.silver.stg_transacciones")

### Validación del parseo

Ninguna fecha con texto de origen puede quedar nula. Si esto falla, apareció un
quinto formato que las guardas no contemplan, y el reporte muestra los ejemplos
concretos para extender la función.

In [0]:
stg = spark.table(f"{CATALOGO}.silver.stg_transacciones")

for columna in ["fecha_inicio", "fecha_fin"]:
    fallidas = stg.filter(
        F.col(columna).isNull() & vacio_a_nulo(F.col(f"{columna}_original")).isNotNull())
    n = fallidas.count()
    if n:
        fallidas.select("id_transaccion", f"{columna}_original").show(20, truncate=False)
    assert n == 0, f"{n} valores de {columna} no se pudieron parsear"

print("Distribución de formatos de origen en fecha_inicio:")
(stg.withColumn("formato",
    F.when(F.col("fecha_inicio_original").rlike(RX_ISO), "ISO")
     .when(F.col("fecha_inicio_original").rlike(RX_BARRA), "DD/MM/AAAA")
     .when(F.col("fecha_inicio_original").rlike(RX_ESPANOL), "DD-mmm-AAAA")
     .when(F.col("fecha_inicio_original").rlike(RX_GUION_NUM), "MM-DD-AAAA")
     .otherwise("(sin clasificar)"))
 .groupBy("formato").count().orderBy(F.desc("count")).show())

Distribución de formatos de origen en fecha_inicio:
+-----------+-----+
|    formato|count|
+-----------+-----+
|        ISO| 2809|
| DD/MM/AAAA|  971|
| MM-DD-AAAA|  750|
|DD-mmm-AAAA|  529|
+-----------+-----+



## Paso 3 — `dim_vehiculo_vigencias`: el SCD tipo 2

**Va antes de la limpieza de transacciones a propósito**: reconstruir la fecha de
un alquiler defectuoso requiere conocer la tarifa vigente en ese momento.

### Regla de vigencia

La primera versión de cada placa no trae `fecha_actualizacion`; su vigencia arranca
en `fecha_ingreso_concesionario`. Las siguientes arrancan en su
`fecha_actualizacion`. El fin de vigencia es el día anterior al inicio de la
siguiente versión, y la versión actual se cierra con la centinela `9999-12-31`,
siguiendo la convención vista en clase.

### Imputación de atributos estables

Se verificó que marca, modelo, año, combustible y costo de mantenimiento por
kilómetro **nunca cambian** entre versiones de una misma placa. Eso habilita dos
imputaciones seguras:

- El año vacío en 60 placas se recupera del sufijo del modelo (`"Taycan - 2025"`).
- El combustible vacío se propaga desde otra versión de la misma placa.

El color **no** se imputa: sí cambia entre versiones, así que propagarlo inventaría
un dato. Queda nulo y marcado con su bandera.

In [0]:
por_placa = W.partitionBy("placa")

carros = (spark.table(f"{CATALOGO}.bronze.carros")
    .withColumn("placa", F.upper(F.trim(F.col("placa"))))
    .withColumn("marca", F.trim(F.col("marca")))
    .withColumn("modelo", F.trim(F.col("modelo")))
    .withColumn("fecha_ingreso_concesionario", parsear_fecha("fecha_ingreso_concesionario"))
    .withColumn("fecha_actualizacion", parsear_fecha("fecha_actualizacion"))
    .withColumn("costo_alquiler_dia", a_numero("costo_alquiler_dia"))
    .withColumn("valor_seguro", a_numero("valor_seguro", "int"))
    .withColumn("costo_mantenimiento_km", a_numero("costo_mantenimiento_km"))
    .withColumn("color", vacio_a_nulo(F.col("color")))
    .withColumn("tipo_combustible", vacio_a_nulo(F.col("tipo_combustible")))
    .drop("_rescued_data"))

carros = (carros
    # Año: castear y, si viene vacío, extraer del sufijo del modelo
    .withColumn("ind_anio_imputado", vacio_a_nulo(F.col("anio")).isNull())
    .withColumn("anio_modelo", F.coalesce(
        a_numero("anio"),
        F.expr(r"try_cast(regexp_extract(modelo, '(\\d{4})\\s*$', 1) as int)")))
    # Combustible: propagar desde otra versión de la misma placa (atributo estable)
    .withColumn("ind_combustible_imputado", F.col("tipo_combustible").isNull())
    .withColumn("tipo_combustible", F.coalesce(
        F.col("tipo_combustible"),
        F.first("tipo_combustible", ignorenulls=True).over(por_placa)))
    # Color: no se imputa, solo se marca
    .withColumn("ind_color_faltante", F.col("color").isNull())
    # Tarifas fuera de escala: se marcan, no se corrigen
    .withColumn("ind_tarifa_fuera_de_rango",
        (F.col("costo_alquiler_dia") < 50000) | (F.col("costo_alquiler_dia") > 1000000))
    .withColumn("fe_vig_ini", F.coalesce(
        F.col("fecha_actualizacion"), F.col("fecha_ingreso_concesionario")))
    .drop("anio"))

ventana_vig = por_placa.orderBy(F.col("fe_vig_ini").asc())

vigencias = (carros
    .withColumn("num_version", F.row_number().over(ventana_vig))
    .withColumn("_siguiente_ini", F.lead("fe_vig_ini").over(ventana_vig))
    .withColumn("fe_vig_fin", F.coalesce(
        F.date_sub(F.col("_siguiente_ini"), 1), F.to_date(F.lit(FECHA_ALTA))))
    .withColumn("es_version_actual", F.col("_siguiente_ini").isNull())
    .drop("_siguiente_ini"))

vigencias.write.mode("overwrite").saveAsTable(f"{CATALOGO}.silver.dim_vehiculo_vigencias")

### Validación del SCD tipo 2

Tres invariantes que deben cumplirse siempre. Si alguna falla, la reconstrucción de
tarifas históricas queda mal y **todo el modelo pierde validez**, porque el valor de
cada alquiler depende de la versión vigente en su fecha de inicio.

In [0]:
vig = spark.table(f"{CATALOGO}.silver.dim_vehiculo_vigencias")
orden_placa = por_placa.orderBy("fe_vig_ini")

# Invariante 1: exactamente una versión actual por placa
multi_actual = (vig.filter(F.col("es_version_actual"))
    .groupBy("placa").count().filter(F.col("count") != 1).count())
assert multi_actual == 0, "Hay placas con cero o varias versiones marcadas como actuales"

# Invariante 2: sin traslapes de vigencia dentro de una placa
traslapes = (vig
    .withColumn("_fin_anterior", F.lag("fe_vig_fin").over(orden_placa))
    .filter(F.col("fe_vig_ini") <= F.col("_fin_anterior")).count())
assert traslapes == 0, "Hay vigencias traslapadas"

# Invariante 3: sin huecos — el inicio sigue al fin anterior por exactamente un día
huecos = (vig
    .withColumn("_fin_anterior", F.lag("fe_vig_fin").over(orden_placa))
    .filter(F.col("_fin_anterior").isNotNull())
    .filter(F.datediff(F.col("fe_vig_ini"), F.col("_fin_anterior")) != 1).count())
assert huecos == 0, "Hay huecos en la línea de tiempo de vigencias"

print(f"SCD2 válido: {vig.count()} versiones sobre {vig.select('placa').distinct().count()} placas")
print(f"Tarifas fuera de rango marcadas: {vig.filter('ind_tarifa_fuera_de_rango').count()}")

SCD2 válido: 1379 versiones sobre 500 placas
Tarifas fuera de rango marcadas: 41


## Paso 4 — Deduplicación

### El hallazgo que simplifica esta etapa

El perfilamiento mostró que los 60 registros con `id_transaccion >= 5000` son
exactamente la cola defectuosa del archivo, y que **53 de ellos duplican un
registro previo**:

| Categoría | Cantidad |
|---|---|
| Copia exacta de un registro anterior | 30 |
| Copia con `valor_total` en 0 | 8 |
| Copia con `valor_total` nulo | 8 |
| Copia con `km_transaccion` nulo | 7 |
| `fecha_fin` sobrescrita, registro único | 7 |

### Por qué se conserva el `id_transaccion` menor

La alternativa intuitiva es escoger la copia "más completa". Se descartó por dos
razones. Primero, en una venta duplicada los dos registros difieren en el cliente y
ninguno es más completo, así que la regla de completitud no decide. Segundo, y más
importante: conservar el `id` menor equivale a **conservar el registro original y
descartar el anexo**, que es la semántica correcta cuando el defecto es una cola mal
apendizada.

El efecto colateral es que los 8 nulos de `valor_total` y los 7 de
`km_transaccion` desaparecen sin una sola regla de imputación, porque todos viven en
las copias descartadas. **Una regla reemplaza cinco.**

### Llave de negocio por proceso

No es la misma para los tres: una placa se compra y se vende una vez, pero se
alquila muchas veces, así que el alquiler necesita además las fechas y la sucursal.

In [0]:
LLAVES_NEGOCIO = {
    "COMPRA": ["placa"],
    "VENTA": ["placa"],
    "ALQUILER": ["placa", "fecha_inicio", "fecha_fin", "nombre_sucursal"],
}

partes = []
for tipo, llave in LLAVES_NEGOCIO.items():
    grupo = W.partitionBy(*llave)
    parte = (stg.filter(F.col("tipo_transaccion") == tipo)
        .withColumn("_rn", F.row_number().over(grupo.orderBy(F.col("id_transaccion").asc())))
        .withColumn("_copias", F.count("*").over(grupo))
        .withColumn("ind_registro_deduplicado", F.col("_copias") > 1)
        .filter(F.col("_rn") == 1)
        .drop("_rn", "_copias"))
    partes.append(parte)

dedup = partes[0].unionByName(partes[1]).unionByName(partes[2])

print(f"Registros de origen: {stg.count()}  ->  tras deduplicar: {dedup.count()}")

Registros de origen: 5059  ->  tras deduplicar: 5006


## Paso 5 — Reparación de las fechas sobrescritas

### Por qué se repara en lugar de poner en cuarentena

Los 7 alquileres con `fecha_fin` anterior a `fecha_inicio` no duplican nada: son
contratos reales. El defecto es sistemático y verificable:

1. En los 7 casos `fecha_fin` es exactamente `fecha_inicio` menos dos días, lo que
   descarta una transposición aleatoria y apunta a una sobrescritura.
2. Al dividir `valor_total` entre la tarifa diaria vigente, el resultado es un
   **entero exacto** en los 7 casos: 6, 12, 5, 13, 8, 9 y 12 días.

El segundo punto es la evidencia decisiva. Si `valor_total` estuviera corrupto, la
división daría decimales; que dé enteros limpios significa que el monto es confiable
y que la duración es reconstruible a partir de él.

### La cautela que exige esta regla

Reconstruir fechas desde `valor_total` sería peligroso como regla general, porque
~10% de los alquileres tiene el monto inconsistente. Por eso la reparación se aplica
**solo cuando la división es exacta**, las filas reparadas quedan marcadas, y
cualquier caso que no cumpla la condición va a cuarentena en lugar de repararse a la
fuerza.

In [0]:
tarifa_vigente = vig.select(
    F.col("placa").alias("_placa"),
    F.col("fe_vig_ini").alias("_ini"),
    F.col("fe_vig_fin").alias("_fin"),
    F.col("costo_alquiler_dia").alias("tarifa_vigente_aplicada"))

con_tarifa = (dedup.join(tarifa_vigente,
        (F.col("placa") == F.col("_placa")) &
        (F.col("fecha_inicio") >= F.col("_ini")) &
        (F.col("fecha_inicio") <= F.col("_fin")), "left")
    .drop("_placa", "_ini", "_fin"))

es_alquiler = F.col("tipo_transaccion") == "ALQUILER"

reparado = (con_tarifa
    .withColumn("_invertida",
        es_alquiler & F.col("fecha_fin").isNotNull() &
        (F.col("fecha_fin") < F.col("fecha_inicio")))
    .withColumn("_dias_implicitos",
        F.when((F.col("tarifa_vigente_aplicada") > 0) &
               (F.col("valor_total") % F.col("tarifa_vigente_aplicada") == 0),
               (F.col("valor_total") / F.col("tarifa_vigente_aplicada")).cast("int")))
    .withColumn("ind_fecha_fin_reparada",
        F.col("_invertida") & F.col("_dias_implicitos").isNotNull())
    .withColumn("ind_cuarentena",
        F.col("_invertida") & F.col("_dias_implicitos").isNull())
    .withColumn("fecha_fin",
        F.when(F.col("ind_fecha_fin_reparada"),
               F.date_add(F.col("fecha_inicio"), F.col("_dias_implicitos")))
         .otherwise(F.col("fecha_fin"))))

## Paso 5b — Banderas de calidad sobre el valor del alquiler

### El caso más delicado del ELT

432 de los 4356 alquileres (9,9%) tienen un `valor_total` que no coincide con
`días × tarifa vigente`. Los patrones detectados, que suman exactamente esos 432:

| Patrón | Casos | ¿Error o regla de negocio? |
|---|---|---|
| Descuento del 10% | 119 | Plausiblemente regla comercial |
| Tarifa de otra versión del vehículo | 107 | Plausiblemente cotización previa al ajuste |
| Dígitos transpuestos | 99 | Error de captura, sin duda |
| Cobra un día menos | 97 | Plausiblemente política de cortesía |
| Sin patrón identificable | 10 | Desconocido |

### Por qué se marca y no se corrige

Tres de los cuatro patrones son indistinguibles de reglas de negocio no
documentadas. Corregirlos sería **sustituir el dato transaccional por un supuesto
nuestro**, y en un caso de consultoría eso es inaceptable: si el descuento del 10%
es real, "corregirlo" infla los ingresos reportados en 119 contratos.

La decisión es conservar el valor de la fuente, calcular el valor teórico al lado y
exponer ambos en gold. El tablero muestra ingreso facturado e ingreso teórico lado a
lado, y la brecha se convierte en un hallazgo cuantificado para VehiAlpes en lugar
de un ajuste que nadie ve.

In [0]:
silver_transacciones = (reparado
    .withColumn("dias_alquiler",
        F.when(es_alquiler, F.datediff(F.col("fecha_fin"), F.col("fecha_inicio"))))
    .withColumn("valor_teorico",
        F.when(es_alquiler, F.col("dias_alquiler") * F.col("tarifa_vigente_aplicada")))
    .withColumn("ind_valor_inconsistente",
        F.coalesce(es_alquiler & (F.col("valor_total") != F.col("valor_teorico")), F.lit(False)))
    .withColumn("ind_fecha_reformateada",
        F.coalesce(~F.col("fecha_inicio_original").rlike(RX_ISO), F.lit(False)))
    .withColumn("ind_kms_atipico", F.coalesce(F.col("kms_recorridos") > 5000, F.lit(False)))
    .drop("_invertida", "_dias_implicitos"))

silver_transacciones.write.mode("overwrite").saveAsTable(f"{CATALOGO}.silver.transacciones")

### Verificación de la reparación

Tras este paso no puede quedar ningún alquiler con la fecha de fin anterior a la de
inicio: o se reparó, o quedó marcado en cuarentena.

In [0]:
sv = spark.table(f"{CATALOGO}.silver.transacciones")

invertidas = sv.filter(
    (F.col("tipo_transaccion") == "ALQUILER") &
    (F.col("fecha_fin") < F.col("fecha_inicio")) & ~F.col("ind_cuarentena")).count()
assert invertidas == 0, f"Quedan {invertidas} alquileres con fechas invertidas sin tratar"

sv.filter("ind_fecha_fin_reparada").select(
    "id_transaccion", "placa", "fecha_inicio_original", "fecha_fin_original",
    "fecha_fin", "dias_alquiler", "tarifa_vigente_aplicada", "valor_total"
).show(truncate=False)

+--------------+------+---------------------+------------------+----------+-------------+-----------------------+-----------+
|id_transaccion|placa |fecha_inicio_original|fecha_fin_original|fecha_fin |dias_alquiler|tarifa_vigente_aplicada|valor_total|
+--------------+------+---------------------+------------------+----------+-------------+-----------------------+-----------+
|5013          |FWU388|19-jun-2025          |17-jun-2025       |2025-07-01|12           |346000                 |4152000    |
|5037          |OHZ176|2025-01-08           |2025-01-06        |2025-01-17|9            |87000                  |783000     |
|5045          |IAY592|2024-09-25           |2024-09-23        |2024-10-03|8            |374000                 |2992000    |
|5029          |LFI440|07-ago-2024          |05-ago-2024       |2024-08-12|5            |316000                 |1580000    |
|5021          |SDW485|07-19-2024           |07-17-2024        |2024-07-31|12           |181000                 |21720

## Paso 6 — `clientes`

### Dos problemas distintos

**60 documentos con dos identificadores.** Se consolidan escogiendo como canónico
el `id_cliente` menor, y se conserva la relación en `id_cliente_unificado`. No se
borra ninguno de los dos registros: ambos tienen transacciones asociadas, y borrar
uno rompería la integridad referencial de esos hechos.

El nombre canónico se escoge como el valor máximo alfabético dentro del documento.
Suena arbitrario, pero en esta fuente funciona: las variantes son del tipo
`"D. Castro"` frente a `"Diana Castro Ortiz"`, y el máximo alfabético selecciona
la forma desarrollada. Se documenta como heurística, no como regla general.

**30 identificadores en transacciones que no existen en el maestro.** Se crean como
miembros inferidos con el nombre marcado como desconocido. La alternativa —descartar
esas 30 transacciones— perdería ingresos reales por un defecto del maestro.

In [0]:
por_documento = W.partitionBy("numero_documento")

clientes = (spark.table(f"{CATALOGO}.bronze.clientes")
    .withColumn("id_cliente", a_numero("id_cliente", "int"))
    .withColumn("numero_documento", F.trim(F.col("numero_documento")))
    .withColumn("nombres", F.trim(F.col("nombres")))
    .withColumn("apellidos", F.trim(F.col("apellidos")))
    .drop("_rescued_data")
    .withColumn("nombre_completo", F.trim(F.concat_ws(" ", F.col("nombres"), F.col("apellidos"))))
    .withColumn("id_cliente_unificado", F.min("id_cliente").over(por_documento))
    .withColumn("_ids_por_documento", F.count("*").over(por_documento))
    .withColumn("ind_registro_duplicado", F.col("_ids_por_documento") > 1)
    .withColumn("nombre_canonico", F.max("nombre_completo").over(por_documento))
    .withColumn("es_miembro_inferido", F.lit(False))
    .select("id_cliente", "numero_documento", "nombres", "apellidos", "nombre_completo",
            "nombre_canonico", "id_cliente_unificado", "ind_registro_duplicado",
            "es_miembro_inferido"))

ids_en_hechos = (sv.filter(F.col("id_cliente").isNotNull())
    .select("id_cliente").distinct())

inferidos = (ids_en_hechos.join(clientes.select("id_cliente"), "id_cliente", "left_anti")
    .withColumn("numero_documento", F.lit(None).cast("string"))
    .withColumn("nombres", F.lit("(desconocido)"))
    .withColumn("apellidos", F.lit(""))
    .withColumn("nombre_completo", F.lit("(desconocido)"))
    .withColumn("nombre_canonico", F.lit("(desconocido)"))
    .withColumn("id_cliente_unificado", F.col("id_cliente"))
    .withColumn("ind_registro_duplicado", F.lit(False))
    .withColumn("es_miembro_inferido", F.lit(True)))

(clientes.unionByName(inferidos)
    .write.mode("overwrite").saveAsTable(f"{CATALOGO}.silver.clientes"))

print(f"Clientes del maestro: {clientes.count()}")
print(f"Miembros inferidos creados: {inferidos.count()}")
print(f"Documentos con más de un identificador: "
      f"{clientes.filter('ind_registro_duplicado').select('numero_documento').distinct().count()}")

Clientes del maestro: 1560
Miembros inferidos creados: 30
Documentos con más de un identificador: 60


## Paso 7 — Conciliación final de la capa

La prueba de que silver no perdió ni inventó filas, y de que gold puede construirse
sobre estos datos sin sorpresas.

In [0]:
print("Conteo por proceso tras deduplicar:")
sv.groupBy("tipo_transaccion").count().orderBy("tipo_transaccion").show()

print("Registros marcados por cada bandera de calidad:")
sv.select(
    F.sum(F.col("ind_registro_deduplicado").cast("int")).alias("deduplicados"),
    F.sum(F.col("ind_fecha_fin_reparada").cast("int")).alias("fechas_reparadas"),
    F.sum(F.col("ind_cuarentena").cast("int")).alias("en_cuarentena"),
    F.sum(F.col("ind_valor_inconsistente").cast("int")).alias("valor_inconsistente"),
    F.sum(F.col("ind_fecha_reformateada").cast("int")).alias("fecha_reformateada"),
    F.sum(F.col("ind_kms_atipico").cast("int")).alias("kms_atipico"),
).show()

Conteo por proceso tras deduplicar:
+----------------+-----+
|tipo_transaccion|count|
+----------------+-----+
|        ALQUILER| 4356|
|          COMPRA|  500|
|           VENTA|  150|
+----------------+-----+

Registros marcados por cada bandera de calidad:
+------------+----------------+-------------+-------------------+------------------+-----------+
|deduplicados|fechas_reparadas|en_cuarentena|valor_inconsistente|fecha_reformateada|kms_atipico|
+------------+----------------+-------------+-------------------+------------------+-----------+
|          53|               7|            0|                432|              2226|         22|
+------------+----------------+-------------+-------------------+------------------+-----------+



In [0]:
# Sin nulos en las medidas clave tras la deduplicación
nulos_valor = sv.filter(F.col("valor_total").isNull()).count()
nulos_km = sv.filter(F.col("km_transaccion").isNull()).count()
print(f"Nulos remanentes — valor_total: {nulos_valor}, km_transaccion: {nulos_km}")
assert nulos_valor == 0 and nulos_km == 0, \
    "Quedan nulos en medidas clave: revisar la regla de deduplicación"

# Integridad referencial hacia clientes y vehículos
huerfanos_cliente = (sv.filter(F.col("id_cliente").isNotNull())
    .join(spark.table(f"{CATALOGO}.silver.clientes").select("id_cliente"),
          "id_cliente", "left_anti").count())
huerfanos_vehiculo = (sv.join(vig.select("placa").distinct(), "placa", "left_anti").count())

assert huerfanos_cliente == 0, f"{huerfanos_cliente} hechos con cliente inexistente"
assert huerfanos_vehiculo == 0, f"{huerfanos_vehiculo} hechos con placa inexistente"

# Toda transacción debe tener una versión de vehículo vigente en su fecha
sin_version = sv.filter(F.col("tarifa_vigente_aplicada").isNull()).count()
print(f"Transacciones sin versión de vehículo vigente en su fecha: {sin_version}")
assert sin_version == 0, "Hay transacciones fuera del rango de vigencia de su vehículo"

print("Integridad referencial verificada")

Nulos remanentes — valor_total: 0, km_transaccion: 0
Transacciones sin versión de vehículo vigente en su fecha: 0
Integridad referencial verificada


In [0]:
for tabla in ["stg_transacciones", "transacciones", "clientes", "dim_vehiculo_vigencias"]:
    spark.sql(f"OPTIMIZE {CATALOGO}.silver.{tabla}")